In [ ]:
from langgraph.prebuilt import tools_condition, ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.types import interrupt, Command
from langchain_core.tools import tool

c:\python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@tool()
def read_file(filename : str) -> str | None:
    """Reads the contents of the given filename and returns the contents"""
    return f"Contents of the file : {filename}" 

In [3]:
@tool()
def delete_file(filename : str) -> str:
    """Deletes the given file"""
    decision = interrupt(f"Shall I Delete File : {filename} ?")
        
    if decision == "yes":
        return f"Deleted file {filename}!"
    else:
        return "Sorry! Deletion cancelled by user!"

In [4]:
tools = [read_file, delete_file]

In [5]:
llm = init_chat_model('gemini-2.5-flash', model_provider='google_genai')
llm_with_tools = llm.bind_tools(tools)
# System message
sys_msg = SystemMessage(content="You are a helpful assistant. Use tools provided whenever needed.")

In [6]:
def call_llm(state: MessagesState):
    response = llm_with_tools.invoke( [sys_msg] + state["messages"])
    return {"messages": [response]}

In [7]:
# Graph
builder = StateGraph(MessagesState)
builder.add_node("call_llm", call_llm)
builder.add_node("tools", ToolNode(tools))
 
builder.add_edge(START, "call_llm")
builder.add_conditional_edges(
    "call_llm",
     tools_condition
)
 
builder.add_edge("tools", "call_llm")
 

In [8]:
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

In [9]:
human_message = HumanMessage(content="Delete file names.txt")

In [10]:
config = {"configurable": {"thread_id": "1"}}
response = graph.invoke({"messages": [human_message]}, config=config)

In [11]:
if response.get("__interrupt__"):
    prompt = response['__interrupt__'][0].value
    decision = input(prompt)
    # Resume the graph with user's input 
    response = graph.invoke(Command(resume = decision), config=config)

In [12]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

Delete file names.txt
================================== Ai Message ==================================
Tool Calls:
  delete_file (5468c1aa-36a1-427a-8e3e-f1fcce107937)
 Call ID: 5468c1aa-36a1-427a-8e3e-f1fcce107937
  Args:
    filename: names.txt
================================= Tool Message =================================
Name: delete_file

Sorry! Deletion cancelled by user!
================================== Ai Message ==================================

I am sorry, I cannot delete the file. The deletion was cancelled by the user.
